In [8]:
#Importar bibliotecas
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = 'notebook'

#Carregar dados
df = pd.read_csv('Telco_Customer_Churn.csv')

#Ver os dados
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [19]:
#Gráfico 1: Cancelamento por Tipo de Contrato
fig1 = px.bar(
    df,
    x = 'Contract',
    color = 'Churn',
    title = 'Cancelamento por Tipo de Contrato', 
    labels = {'Contract': 'Tipo de Contrato', 'count:': 'Quantidade de Clientes'},
    barmode = 'group',
    color_discrete_map = {'No': '#66b3ff', 'Yes': '#ff9999'}
)

fig1.show()

In [14]:
#Gráfico 2:
fig2 = px.histogram(
    df,
    x = 'tenure',
    color = 'Churn',
    title = 'Distribuição de Tempo como Cliente por cancelamento',
    labels = {'tenure': 'Meses como Cliente', 'count': 'Quantidade'},
    nbins = 20,
    color_discrete_map = {'No': '#66b3ff', 'Yes': '#ff9999'}  
)
fig2.show()

In [16]:
#Gráfico 3: Distribuição do valor da mensalidade por cancelamento
fig3 = px.box(
    df,
    x = 'Churn',
    y = 'MonthlyCharges',
    color = 'Churn',
    title = 'Valor da Mensalidade por Clientes (Cancelou vs Não Cancelou)',
    labels = {'Churn': 'Cancelou', 'MonthlyCharges': 'Valor da Mensalidade (R$)'},
    color_discrete_map = {'No': '#66b3ff', 'Yes':'#ff9999'}
)
fig3.show()

In [29]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# Carregar os dados
df = pd.read_csv('Telco_Customer_Churn.csv')

# Preparar dados para o gráfico de barras (Contrato vs Churn)
contract_counts = df.groupby(['Contract', 'Churn']).size().reset_index(name='count')

# Criar o dashboard com 2 linhas e 2 colunas
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Cancelamento por Tipo de Contrato',
        'Tempo como Cliente (Meses)',
        'Valor da Mensalidade',
        'Serviços Contratados'
    ),
    specs=[[{'type': 'bar'}, {'type': 'histogram'}],
           [{'type': 'box'}, {'type': 'bar'}]]
)

# GRÁFICO 1: Barras - Cancelamento por Contrato (linha 1, coluna 1)
cores_churn = {'No': '#66b3ff', 'Yes': '#ff9999'}
for churn in ['No', 'Yes']:
    dados = contract_counts[contract_counts['Churn'] == churn]
    fig.add_trace(
        go.Bar(
            x=dados['Contract'],
            y=dados['count'],
            name=churn,
            marker_color=cores_churn[churn],
            text=dados['count'],
            textposition='outside'
        ),
        row=1, col=1
    )

# GRÁFICO 2: Histograma - Tempo como Cliente (linha 1, coluna 2)
for churn in ['No', 'Yes']:
    dados = df[df['Churn'] == churn]
    fig.add_trace(
        go.Histogram(
            x=dados['tenure'],
            name=churn,
            marker_color=cores_churn[churn],
            opacity=0.7,
            nbinsx=20
        ),
        row=1, col=2
    )

# GRÁFICO 3: Boxplot - Mensalidade (linha 2, coluna 1)
for churn in ['No', 'Yes']:
    dados = df[df['Churn'] == churn]
    fig.add_trace(
        go.Box(
            y=dados['MonthlyCharges'],
            name=churn,
            marker_color=cores_churn[churn],
            boxmean='sd'  # mostra média e desvio padrão
        ),
        row=2, col=1
    )

# GRÁFICO 4: Barras - Serviços Contratados (linha 2, coluna 2)
servicos = ['PhoneService', 'InternetService', 'OnlineSecurity', 'TechSupport']
servicos_counts = {}
for servico in servicos:
    servicos_counts[servico] = df[df[servico] != 'No']['Churn'].value_counts().get('Yes', 0)

fig.add_trace(
    go.Bar(
        x=list(servicos_counts.keys()),
        y=list(servicos_counts.values()),
        name='Cancelaram',
        marker_color='#ff9999',
        text=list(servicos_counts.values()),
        textposition='outside'
    ),
    row=2, col=2
)

# Atualizar layout do dashboard
fig.update_layout(
    height=800,
    width=1000,
    title_text='<b>Dashboard de Análise de Cancelamento de Clientes</b>',
    title_font_size=20,
    showlegend=True,
    legend_title_text='Cancelou?',
    template='plotly_white'
)

# Ajustar eixos
fig.update_xaxes(title_text='Tipo de Contrato', row=1, col=1)
fig.update_xaxes(title_text='Meses como Cliente', row=1, col=2)
fig.update_xaxes(title_text='', row=2, col=1)
fig.update_xaxes(title_text='Serviço', row=2, col=2)

fig.update_yaxes(title_text='Quantidade', row=1, col=1)
fig.update_yaxes(title_text='Quantidade', row=1, col=2)
fig.update_yaxes(title_text='Valor (R$)', row=2, col=1)
fig.update_yaxes(title_text='Cancelamentos', row=2, col=2)

# Salvar como HTML (para compartilhar)
fig.write_html('dashboard_cancelamento_profissional.html')

# Exibir
fig.show()

In [1]:
with open('README.md', 'w', encoding='utf-8') as arquivo:
    arquivo.write('# Dashboard Interativo - Cancelamento de Clientes\n\n')
    arquivo.write('![Python](https://img.shields.io/badge/Python-3.14-blue)\n')
    arquivo.write('![Plotly](https://img.shields.io/badge/Plotly-5.18-orange)\n')
    arquivo.write('![Status](https://img.shields.io/badge/Status-Concluído-brightgreen)\n\n')
    
    arquivo.write('## Sobre o Projeto\n')
    arquivo.write('Dashboard interativo criado com **Plotly** para análise de cancelamento de clientes de uma empresa de telecomunicações.\n\n')
    
    arquivo.write('## Visualize o Dashboard\n')
    arquivo.write('Clique aqui para abrir o dashboard interativo: [dashboard_cancelamento_profissional.html](dashboard_cancelamento_profissional.html)\n\n')
    
    arquivo.write('## Gráficos Incluídos\n\n')
    arquivo.write('| Gráfico | Insight |\n')
    arquivo.write('|---------|---------|\n')
    arquivo.write('| Cancelamento por Contrato | Contrato mensal tem maior taxa de cancelamento |\n')
    arquivo.write('| Tempo como Cliente | Clientes que cancelam têm menos tempo de casa |\n')
    arquivo.write('| Valor da Mensalidade | Clientes que cancelam pagam mensalidades mais altas |\n')
    arquivo.write('| Serviços Contratados | Serviços de segurança reduzem cancelamento |\n\n')
    
    arquivo.write('## Como Executar\n\n')
    arquivo.write('```bash\n')
    arquivo.write('pip install pandas plotly\n')
    arquivo.write('jupyter notebook dashboard_cancelamento.ipynb\n')
    arquivo.write('```\n\n')
    
    arquivo.write('## Tecnologias\n')
    arquivo.write('- Python\n')
    arquivo.write('- Pandas (manipulação de dados)\n')
    arquivo.write('- Plotly (gráficos interativos)\n')
    arquivo.write('- Jupyter Notebook\n')

print("README criado com sucesso!")

README criado com sucesso!
